# fractional-stride-zero-insertion — worked example 2: Zero Insertion for Stride-2 ConvTranspose in 2-D

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `fractional-stride-zero-insertion`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

For 2-D inputs, zero insertion applies independently along both spatial dimensions. Given an input of shape `(B, C, H, W)` and stride `s`, the zero-inserted output has shape `(B, C, (H-1)*s+1, (W-1)*s+1)`. The assignment `output[:, :, ::s, ::s] = input` scatters the original pixels into every `s`-th position in both dimensions simultaneously.

## Worked solution

We implement 2-D zero insertion for stride 2 and verify by comparing a subsequent stride-1 conv against the real `F.conv_transpose2d(stride=2)`.

**Input:** `(1, 1, 3, 3)` — one sample, one channel, 3×3 spatial.

**Output shape:** `(1, 1, (3-1)*2+1, (3-1)*2+1) = (1, 1, 5, 5)`.

**Assignment:** `y[:, :, ::2, ::2] = x` puts the 9 original pixels at the odd grid positions; the 16 remaining positions are zeros.

**Equivalence check:** A stride-1 transposed conv on the zero-inserted tensor should produce the same result as a stride-2 transposed conv on the original. This is the mathematical definition of what ConvTranspose2d with stride>1 computes.

In [ ]:
import torch as t
import torch.nn.functional as F

t.manual_seed(15)

def zero_insert_2d(x: t.Tensor, s: int) -> t.Tensor:
    B, C, H, W = x.shape
    H_out = (H - 1) * s + 1
    W_out = (W - 1) * s + 1
    y = t.zeros(B, C, H_out, W_out, dtype=x.dtype)
    y[:, :, ::s, ::s] = x
    return y

# s=2, small 3x3 input
x = t.randn(1, 1, 3, 3)
zi = zero_insert_2d(x, s=2)
print(f"Input shape:  {x.shape}")
print(f"Zero-inserted shape: {zi.shape}  (expected (1,1,5,5))")
assert zi.shape == (1, 1, 5, 5)

# Verify only the ::2 positions are non-zero
assert t.allclose(zi[:, :, ::2, ::2], x)
assert zi[:, :, 1, :].abs().sum() == 0
assert zi[:, :, :, 1].abs().sum() == 0

# Equivalence: stride-1 convT on zero-inserted == stride-2 convT on x
weight = t.randn(1, 1, 3, 3)

# Method A: zero-insert then stride-1 convT
out_A = F.conv_transpose2d(zi, weight, stride=1)

# Method B: stride-2 convT directly (with padding to match shapes)
out_B = F.conv_transpose2d(x, weight, stride=2)

print(f"Method A shape: {out_A.shape}")
print(f"Method B shape: {out_B.shape}")
assert out_A.shape == out_B.shape, f"{out_A.shape} != {out_B.shape}"
assert t.allclose(out_A, out_B, atol=1e-5), "Zero-insert equivalence failed"
print("Zero-insert → stride-1 convT equals direct stride-2 convT.")